In [1]:
# ==========================================
# 1. Import Libraries
# ==========================================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from lightgbm import LGBMClassifier


# ==========================================
# 2. Create Imbalanced Classification Dataset
# ==========================================

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    weights=[0.90, 0.10],
    random_state=42
)


# ==========================================
# 3. Convert into DataFrame
# ==========================================

X = pd.DataFrame(
    X,
    columns=[f"Feature_{i}" for i in range(1, 11)]
)

y = pd.Series(y, name="Target")


# ==========================================
# 4. Check Class Imbalance
# ==========================================

print("Class Counts:")
print(y.value_counts())

print("\nClass Percentages:")
print(y.value_counts(normalize=True) * 100)


# ==========================================
# 5. Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ==========================================
# 6. Create LightGBM Classifier
# ==========================================

model = LGBMClassifier(
    objective="binary",
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,

    is_unbalance=True,

    random_state=42
)


# ==========================================
# 7. Train Model
# ==========================================

model.fit(X_train, y_train)


# ==========================================
# 8. Predict Class
# ==========================================

y_pred = model.predict(X_test)

print("\nPredicted Classes:")
print(y_pred[:10])


# ==========================================
# 9. Predict Probability
# ==========================================

y_prob = model.predict_proba(X_test)[:, 1]

print("\nPredicted Probabilities:")
print(y_prob[:10])


# ==========================================
# 10. Accuracy
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)


# ==========================================
# 11. ROC-AUC
# ==========================================

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)


# ==========================================
# 12. Confusion Matrix
# ==========================================

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


# ==========================================
# 13. Classification Report
# ==========================================

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# ==========================================
# 14. Custom Threshold
# ==========================================

threshold = 0.30

y_pred_custom = (y_prob >= threshold).astype(int)

print("\nCustom Threshold:", threshold)

print("\nConfusion Matrix - Custom Threshold:")
print(confusion_matrix(y_test, y_pred_custom))

print("\nClassification Report - Custom Threshold:")
print(classification_report(y_test, y_pred_custom))

Class Counts:
Target
0    4478
1     522
Name: count, dtype: int64

Class Percentages:
Target
0    89.56
1    10.44
Name: proportion, dtype: float64
[LightGBM] [Info] Number of positive: 418, number of negative: 3582
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000883 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.104500 -> initscore=-2.148195
[LightGBM] [Info] Start training from score -2.148195

Predicted Classes:
[0 0 1 0 0 0 0 1 0 0]

Predicted Probabilities:
[0.28396249 0.2260606  0.91575548 0.09917109 0.07961756 0.01021447
 0.13749073 0.84137958 0.28903507 0.01326144]

Accuracy: 0.94
ROC-AUC: 0.9525347699175823

Confusion Matrix:
[[861  35]
 [ 25  79]]

Classification Report:
              precision    recall  f1-score   support

           0   